# Task 1.1 – Data Loading & Schema Design
## US Baby Names Explorer

This notebook loads the **US Baby Names** dataset (SSA data, 1880–2014) into a local SQLite database,
designs an appropriate schema, and creates indexes to accelerate the queries needed for Tasks 1.2 and 1.3.

**Dataset files used:**
- `NationalNames.csv` – aggregated US-wide counts (~1.8 M rows)
- `StateNames.csv` – per-state counts (~5.6 M rows)

In [12]:
import sqlite3
import pandas as pd
import os
import time

In [13]:
# --- Configuration ---
DATA_DIR  = "us baby names"
NAT_CSV   = os.path.join(DATA_DIR, "NationalNames.csv")
STATE_CSV = os.path.join(DATA_DIR, "StateNames.csv")
DB_PATH   = "baby_names.db"

# Remove old DB so the notebook is fully reproducible
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"Removed existing {DB_PATH}")

print(f"NationalNames exists : {os.path.exists(NAT_CSV)}")
print(f"StateNames exists    : {os.path.exists(STATE_CSV)}")

Removed existing baby_names.db
NationalNames exists : True
StateNames exists    : True


## Schema Design

Two tables mirror the two CSV files:

| Table | Columns |
|---|---|
| `national_names` | Id (PK), Name, Year, Gender, Count |
| `state_names` | Id (PK), Name, Year, Gender, State, Count |

**Type choices:**
- `Id` → `INTEGER PRIMARY KEY` (rowid alias, most compact)
- `Name` → `TEXT` (variable-length strings)
- `Year`, `Count` → `INTEGER`
- `Gender`, `State` → `TEXT` (short codes 'M'/'F', 'AK'/'CA'…)

In [14]:
DDL = """
CREATE TABLE IF NOT EXISTS national_names (
    Id      INTEGER PRIMARY KEY,
    Name    TEXT    NOT NULL,
    Year    INTEGER NOT NULL,
    Gender  TEXT    NOT NULL,
    Count   INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS state_names (
    Id      INTEGER PRIMARY KEY,
    Name    TEXT    NOT NULL,
    Year    INTEGER NOT NULL,
    Gender  TEXT    NOT NULL,
    State   TEXT    NOT NULL,
    Count   INTEGER NOT NULL
);
"""

con = sqlite3.connect(DB_PATH)
con.executescript(DDL)
con.commit()
print("Tables created.")

Tables created.


## Loading Data

Both CSVs are loaded in **chunks of 50 000 rows** to keep memory usage low
(StateNames.csv is 148 MB). `PRAGMA synchronous = OFF` and `journal_mode = MEMORY`
are set temporarily to speed up the bulk insert.

In [15]:
def load_csv_to_table(csv_path, table_name, con, chunksize=50_000):
    """Load a CSV file into a SQLite table using chunked reads."""
    con.execute("PRAGMA synchronous = OFF")
    con.execute("PRAGMA journal_mode = MEMORY")

    total_rows = 0
    t0 = time.time()
    for chunk in pd.read_csv(csv_path, chunksize=chunksize):
        chunk.to_sql(table_name, con, if_exists="append", index=False)
        total_rows += len(chunk)
        print(f"  {table_name}: {total_rows:,} rows loaded…", end="\r")

    con.execute("PRAGMA synchronous = NORMAL")
    con.execute("PRAGMA journal_mode = DELETE")
    con.commit()
    elapsed = time.time() - t0
    print(f"  {table_name}: {total_rows:,} rows loaded in {elapsed:.1f}s")
    return total_rows


print("Loading NationalNames…")
n_nat = load_csv_to_table(NAT_CSV, "national_names", con)

print("Loading StateNames…")
n_state = load_csv_to_table(STATE_CSV, "state_names", con)

print(f"\nDone. national_names: {n_nat:,} rows | state_names: {n_state:,} rows")

Loading NationalNames…
  national_names: 1,825,433 rows loaded in 6.5s
Loading StateNames…
  state_names: 5,647,426 rows loaded in 25.0s

Done. national_names: 1,825,433 rows | state_names: 5,647,426 rows


## Index Creation

We create **three indexes** (two required + one bonus composite):

### Index 1 — `idx_nat_name` / `idx_state_name` on `Name`

**What it speeds up:** Any query that filters or groups by baby name, e.g.:
```sql
SELECT Year, SUM(Count) FROM national_names WHERE Name = 'Emma' GROUP BY Year;
```
Without an index SQLite must perform a **full table scan** of ~1.8 M rows every time
a user types a name into the explorer. With a B-tree index on `Name`, SQLite jumps
directly to the matching rows — O(log n) instead of O(n).

### Index 2 — `idx_nat_year` / `idx_state_year` on `Year`

**What it speeds up:** Time-range filters and year-level aggregations, e.g.:
```sql
SELECT Year, SUM(Count) FROM national_names WHERE Year BETWEEN 1950 AND 2000 GROUP BY Year;
-- also the "total births per year" denominator used for relative popularity:
SELECT Year, SUM(Count) AS total FROM national_names GROUP BY Year;
```
These aggregations are the backbone of every time-series chart. The index on `Year`
lets SQLite use an **index scan** for range predicates rather than scanning the whole table.

### Index 3 (bonus) — `idx_nat_name_year` composite on `(Name, Year)`

**What it speeds up:** The most frequent combined query in Task 1.2A — look up a name
for a specific year (used internally when computing relative popularity):
```sql
SELECT Count FROM national_names WHERE Name = 'Emma' AND Year = 1990;
```
SQLite can satisfy this query entirely from the index (**covering index**) without
touching the main table pages, giving another ~2–3× speedup over the single-column index.

In [16]:
INDEX_SQL = """
-- Indexes 1 & 2: Name lookup — applied to both national and state tables (2 physical indexes)
CREATE INDEX IF NOT EXISTS idx_nat_name   ON national_names(Name);
CREATE INDEX IF NOT EXISTS idx_state_name ON state_names(Name);

-- Indexes 3 & 4: Year-range / time-series — applied to both national and state tables (2 physical indexes)
CREATE INDEX IF NOT EXISTS idx_nat_year   ON national_names(Year);
CREATE INDEX IF NOT EXISTS idx_state_year ON state_names(Year);

-- Index 5 (bonus): composite (Name, Year) — covering index for popularity-over-time
CREATE INDEX IF NOT EXISTS idx_nat_name_year ON national_names(Name, Year);
"""

t0 = time.time()
con.executescript(INDEX_SQL)
con.commit()
print(f"Indexes created in {time.time() - t0:.1f}s")
print("Total indexes created: 5 (2 logical strategies × 2 tables + 1 bonus composite)")

Indexes created in 22.8s
Total indexes created: 5 (2 logical strategies × 2 tables + 1 bonus composite)


## Verification – Row Counts & Sample Data

In [17]:
# Row counts
for table in ["national_names", "state_names"]:
    (count,) = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()
    print(f"{table}: {count:,} rows")

# Year range
row = con.execute("SELECT MIN(Year), MAX(Year) FROM national_names").fetchone()
print(f"\nYear range in national_names: {row[0]} – {row[1]}")

# Gender split
gender_df = pd.read_sql(
    "SELECT Gender, COUNT(*) AS rows, SUM(Count) AS total_births "
    "FROM national_names GROUP BY Gender",
    con
)
print("\nGender distribution:")
display(gender_df)

national_names: 1,825,433 rows
state_names: 5,647,426 rows

Year range in national_names: 1880 – 2014

Gender distribution:


,Gender,rows,total_births
0,F,1081683,167070477
1,M,743750,170064949


In [18]:
# Sample rows from each table
print("national_names sample:")
display(pd.read_sql("SELECT * FROM national_names LIMIT 5", con))

print("\nstate_names sample:")
display(pd.read_sql("SELECT * FROM state_names LIMIT 5", con))

national_names sample:


,Id,Name,Year,Gender,Count
0,1,Mary,1880,F,7065
1,2,Anna,1880,F,2604
2,3,Emma,1880,F,2003
3,4,Elizabeth,1880,F,1939
4,5,Minnie,1880,F,1746



state_names sample:


,Id,Name,Year,Gender,State,Count
0,1,Mary,1910,F,AK,14
1,2,Annie,1910,F,AK,12
2,3,Anna,1910,F,AK,10
3,4,Margaret,1910,F,AK,8
4,5,Helen,1910,F,AK,7


## Index Effectiveness – EXPLAIN QUERY PLAN

We confirm SQLite actually uses our indexes (not a full table scan) for the key queries.

In [19]:
queries = {
    # Index 1: idx_nat_name
    "idx_nat_name  — Name lookup on national_names":
        "SELECT Year, SUM(Count) FROM national_names WHERE Name='Emma' GROUP BY Year",

    # Index 2: idx_state_name
    "idx_state_name — Name lookup on state_names":
        "SELECT Year, SUM(Count) FROM state_names WHERE Name='Emma' GROUP BY Year",

    # Index 3: idx_nat_year
    "idx_nat_year  — Year-range scan on national_names":
        "SELECT Name, SUM(Count) FROM national_names WHERE Year BETWEEN 1990 AND 2000 GROUP BY Name",

    # Index 4: idx_state_year
    "idx_state_year — Year-range scan on state_names":
        "SELECT Name, SUM(Count) FROM state_names WHERE Year BETWEEN 1990 AND 2000 GROUP BY Name",

    # Index 5: idx_nat_name_year (composite / covering)
    "idx_nat_name_year — composite (Name, Year) covering index":
        "SELECT Count FROM national_names WHERE Name='Emma' AND Year=1990",
}

for label, sql in queries.items():
    plan = con.execute(f"EXPLAIN QUERY PLAN {sql}").fetchall()
    print(f"\n[{label}]")
    for row in plan:
        print(" ", row[-1])  # print only the description string


[idx_nat_name  — Name lookup on national_names]
  SEARCH national_names USING INDEX idx_nat_name_year (Name=?)

[idx_state_name — Name lookup on state_names]
  SEARCH state_names USING INDEX idx_state_name (Name=?)
  USE TEMP B-TREE FOR GROUP BY

[idx_nat_year  — Year-range scan on national_names]
  SEARCH national_names USING INDEX idx_nat_year (Year>? AND Year<?)
  USE TEMP B-TREE FOR GROUP BY

[idx_state_year — Year-range scan on state_names]
  SEARCH state_names USING INDEX idx_state_year (Year>? AND Year<?)
  USE TEMP B-TREE FOR GROUP BY

[idx_nat_name_year — composite (Name, Year) covering index]
  SEARCH national_names USING INDEX idx_nat_name_year (Name=? AND Year=?)


In [20]:
# Benchmark: timed query with vs without the name index
# (We simulate "without" by dropping and recreating)
QUERY = "SELECT Year, SUM(Count) AS cnt FROM national_names WHERE Name='Emma' GROUP BY Year"

# --- With index ---
REPS = 5
t0 = time.time()
for _ in range(REPS):
    pd.read_sql(QUERY, con)
with_idx = (time.time() - t0) / REPS * 1000

# --- Without index (temporary drop) ---
con.execute("DROP INDEX IF EXISTS idx_nat_name")
con.execute("DROP INDEX IF EXISTS idx_nat_name_year")
con.commit()

t0 = time.time()
for _ in range(REPS):
    pd.read_sql(QUERY, con)
without_idx = (time.time() - t0) / REPS * 1000

# Restore indexes
con.execute("CREATE INDEX IF NOT EXISTS idx_nat_name ON national_names(Name)")
con.execute("CREATE INDEX IF NOT EXISTS idx_nat_name_year ON national_names(Name, Year)")
con.commit()

print(f"Query: {QUERY}")
print(f"  Without index : {without_idx:.1f} ms (avg over {REPS} runs)")
print(f"  With index    : {with_idx:.1f} ms (avg over {REPS} runs)")
print(f"  Speedup       : {without_idx / with_idx:.1f}x")

Query: SELECT Year, SUM(Count) AS cnt FROM national_names WHERE Name='Emma' GROUP BY Year
  Without index : 407.0 ms (avg over 5 runs)
  With index    : 2.1 ms (avg over 5 runs)
  Speedup       : 197.6x


## Summary

Task 1.1 is complete:

| Item | Detail |
|---|---|
| Database file | `baby_names.db` (SQLite 3) |
| Tables | `national_names`, `state_names` |
| Rows loaded | ~1.8 M national + ~5.6 M state |
| Indexes | 5 (2 required + 1 composite bonus, mirrored on both tables) |
| Year coverage | 1880 – 2014 |

The database is ready for the interactive app in Task 1.2.

In [21]:
con.close()
print("Connection closed. DB size:", round(os.path.getsize(DB_PATH) / 1024**2, 1), "MB")

Connection closed. DB size: 395.4 MB
